In [5]:
import pandas as pd
import pandas.io.sql as sqlio
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
import sqlalchemy
import psycopg2 as ps
from sqlalchemy import create_engine
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier
import lightgbm as lgb
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score


In [2]:
from connexion import Connexion as cnn
fa = """SELECT * FROM public."Fact_Activity";"""
fo= """SELECT * FROM public."Fact_Orders";"""
da= """SELECT * FROM public."Dim_Accident";"""
dc= """SELECT * FROM public."Dim_Customer";"""
dd= """SELECT * FROM public."Dim_Date";"""
de= """SELECT * FROM public."Dim_Event";"""
dm= """SELECT * FROM public."Dim_Meteo";"""
dp= """SELECT * FROM public."Dim_Payment";"""
dpp= """SELECT * FROM public."Dim_Products";"""
du= """SELECT * FROM public."Dim_User";"""

dim_acc = cnn.connect_to_dw(da)
dim_acc = pd.DataFrame(dim_acc)
dim_cus = cnn.connect_to_dw(dc)
dim_cus = pd.DataFrame(dim_cus)
dim_date = cnn.connect_to_dw(dd)
dim_date = pd.DataFrame(dim_date)
dim_event = cnn.connect_to_dw(de)
dim_event = pd.DataFrame(dim_event)
dim_meteo = cnn.connect_to_dw(dm)
dim_meteo = pd.DataFrame(dim_meteo)
dim_payment = cnn.connect_to_dw(dp)
dim_payment = pd.DataFrame(dim_payment)
dim_product = cnn.connect_to_dw(dpp)
dim_product = pd.DataFrame(dim_product)
dim_user = cnn.connect_to_dw(du)
dim_user = pd.DataFrame(dim_user)
fact_activity = cnn.connect_to_dw(fa)
fact_activity = pd.DataFrame(fact_activity)
fact_orders = cnn.connect_to_dw(fo)
fact_orders = pd.DataFrame(fact_orders)



Connexion class created successfully!
       Activity_Pk  Accident_FK  Date_Accident_FK  Event_FK  Date_Event_FK  \
0          1100375            0                 0   1040157         174468   
1          1100376            0                 0   1040160         302628   
2          1100377            0                 0   1040160         401988   
3          1100378            0                 0   1040160         286788   
4          1100379            0                 0   1040157         132708   
...            ...          ...               ...       ...            ...   
21518      1121893            0                 0   1040163         471108   
21519      1121894           10            125508   1040163         387588   
21520      1121895            0                 0   1040159         188868   
21521      1121896            0                 0   1040163         353028   
21522      1121897            0                 0   1040157         387588   

       Date_FK  Meteo_FK 

In [3]:
def shuufle_data(data):
    original_dtypes=data.dtypes
    dim_date_array = data.values

    # Shuffle the rows
    np.random.shuffle(dim_date_array)

    # Convert back to DataFrame
    shuffled_dim_date = pd.DataFrame(dim_date_array, columns=data.columns)
    shuffled_dim_date = shuffled_dim_date.astype(original_dtypes)

    data=shuffled_dim_date
    return data

In [6]:


# Load relevant data for customer prediction
df_review = pd.merge(fact_activity, dim_date, left_on='Date_FK', right_on='Date_PK')
df_review = df_review[['Year', 'Month', 'Day', 'Event_FK', 'Customer_Reviews','Temperature','Accident_Rate']]
df_review["Temperature"] = df_review["Temperature"].str.replace(' C/', '').str.replace('C', '').str.replace('/', '')
df_review["Temperature"] = df_review["Temperature"].astype('string').str[:2].astype('int')

# Define conditions for customer sentiment categories , we embed the sentiment in the data
conditions = [
    (df_review['Customer_Reviews'] <= 5.0) & (df_review['Customer_Reviews'] >= 4.0),
    (df_review['Customer_Reviews'] < 4.0) & (df_review['Customer_Reviews'] >= 2.0),
    (df_review['Customer_Reviews'] < 2.0)
]
categories = [3, 2, 1]
# Apply conditions to assign sentiment categories
df_review["feeling"] = np.select(conditions, categories, default=np.nan)
df_review['feeling'] = df_review['feeling'].astype('int')
shuufle_data(df_review)
print(df_review.head())
# Extract features and target variable
X_review = df_review[['Year', 'Month', 'Day', 'Event_FK', 'Customer_Reviews','Temperature','Accident_Rate']]
y_review = df_review['feeling']

# Initialize the Logistic Regression and SGDClassifier
logistic_model = LogisticRegression()
sgd_model = GaussianNB()

# Perform 5-fold cross-validation for Logistic Regression
logistic_scores = cross_val_score(logistic_model, X_review, y_review, cv=20)

# Perform 5-fold cross-validation for SGDClassifier
sgd_scores = cross_val_score(sgd_model, X_review, y_review, cv=20)

# Print the mean accuracy of each classifier
print("Logistic Regression Mean Accuracy:", logistic_scores.mean()*100)
print("SGDClassifier Mean Accuracy:", sgd_scores.mean()*100)

   Year  Month  Day  Event_FK  Customer_Reviews  Temperature  Accident_Rate  \
0  2023      5    2   1040157               4.8           15            0.0   
1  2023      7   30   1040160               3.6           26            0.0   
2  2023     10    7   1040160               4.5           19            0.0   
3  2023      7   19   1040160               4.8           30            0.0   
4  2023      4    3   1040157               3.7           12            0.0   

   feeling  
0        3  
1        2  
2        3  
3        3  
4        2  


c:\ProjetBI\venv\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\ProjetBI\venv\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optim

Logistic Regression Mean Accuracy: 97.71381936606227
SGDClassifier Mean Accuracy: 99.86989279045126
